# Importing Libraries

In [10]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier


LOADING THE DATASET

In [11]:
customer_transaction_data = pd.read_csv(r"C:\Users\pc\Downloads\Fraudulent_Transaction_Detection_Using_Machine_Learning\dataset\FinLora_Customer_Transaction_Dataset.csv")
customer_transaction_data.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [12]:
# checking for number of rows and columns
customer_transaction_data.shape

(11400, 26)

In [13]:
# checking for all the columns in the dataset
customer_transaction_data.columns

Index(['transaction_id', 'customer_id', 'timestamp', 'home_country',
       'source_currency', 'dest_currency', 'channel', 'amount_src',
       'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'device_id',
       'new_device', 'ip_address', 'ip_country', 'location_mismatch',
       'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score',
       'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud'],
      dtype='str')

checking of missing values in each of variables we can confirm that:
timestamp has 29 missing values

amount_usd has 305 missing values
fess has 295 missing values
ip_address has 305 missing values
ip_country has 301 missing values
kyc_tier has 300 missing values
device_trust_score has 295 missing values

In [14]:
# checking for missing data
customer_transaction_data.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [15]:
customer_transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [ ]:
# checking transactions numbers made
customer_transaction_data['amount_src'].value_counts(dropna=False).head(20)

amount_src
100.0     15
102.29     5
115.29     4
54.74      4
167.16     4
82.25      4
128.93     4
261.23     4
71.78      4
102.37     4
105.2      4
121.3      4
119.15     4
250.19     4
58.93      4
94.34      4
144.29     4
165.91     3
115.25     3
111.18     3
Name: count, dtype: int64

summary / observations

The composition of the dataset is 11400 rows and 26 columns
All fieldnames follows the standard variables naming  conventions
An initial inspection of the dataset revealed inconsistence in the data types of certain variables. specifically the timestamp variablesis currently stored as an object data type instead of the appropriate datetime format, which is neccessary for accurate time-based analysis, similarly the amount_src variable is also stored as an  object data type , whereas it should ideally be represented as a numeric format integer or floats to allow for proper numerical computations and analysis. These inconsistences will need to be corrected during the data proprocessing stage.

An assessement of missing values across the dataset revealed that several variables contain incomplete records, specifically the timestamp variables contains 29 missing values ip_country has 301 misssing values and kyc_tier contains 300 misssing values. These missing values will require appropriate data cleaning and preprocessing to ensure the reliability and accuracy of subsequent analysis and modelling.
The target variables is_fraud is highly imbalanced, with appropriate 91%  of transaction classified as  legitimate and only 9% as identified as fraudulent.
This imbalance introduce a risk of model bias towards the majority class where machine learning algorithms may prioritize predicting legitimate transaction while failing to detect fraudulent ones. As a result relying solely on accuracy could produce misleading results To mitigate this issues techniques such as resampling methods and appropriate evaluation metrics should be used to ensure the model effectively identifies  fraudulent transactions.

In [17]:
# checking for label imbalance
customer_transaction_data['is_fraud'].value_counts(normalize=True).reset_index()

,is_fraud,proportion
0,0,0.912544
1,1,0.087456


DATA CLEANING AND PREPARATION

In [18]:
# converting data types of timestamp to datetime
customer_transaction_data['timestamp'] = pd.to_datetime(customer_transaction_data['timestamp'],errors='coerce')

# converting data type of income_usd to float
customer_transaction_data['amount_src'] = pd.to_numeric(customer_transaction_data['amount_src'], errors='coerce')

In [19]:
# checking the dataset again to confrim the data type
customer_transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  str                
 1   customer_id                11400 non-null  str                
 2   timestamp                  11339 non-null  datetime64[us, UTC]
 3   home_country               11400 non-null  str                
 4   source_currency            11400 non-null  str                
 5   dest_currency              11400 non-null  str                
 6   channel                    11400 non-null  str                
 7   amount_src                 11396 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  device_id    